In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
DIRECTORY = pathlib.Path("~/Box/dsi-core/11th-hour/good-food-purchasing/nov2025-dataset").expanduser()

In [3]:
columns = [
    "Food Product Group",
    "Food Product Category",
    "Primary Food Product Category",
    "Basic Type",
    "Sub-Type",
    "Flavor/Cut",
    "Shape",
    "Skin",
    "Seed/Bone",
    "Processing",
    "Cooked/Cleaned",
    "WG/WGR",
    "Dietary Concern",
    "Additives",
    "Dietary Accommodation",
    "Frozen",
    "Packaging",
    "Commodity",
]
column_data = {f"want_{c}": [] for c in columns} | {f"got_{c}": [] for c in columns}

with open(DIRECTORY / "test-results-try3.jsonl") as file:
    for line in file:
        data = json.loads(line)
        if data["status_code"] != 200 or data["deformed_output"] is not None:
            continue
        expected_output = data["expected_output"]
        actual_output = data["actual_output"]
        for c in columns:
            column_data[f"want_{c}"].append(expected_output.get(c, None))
            column_data[f"got_{c}"].append(actual_output.get(c, None))

df = pd.DataFrame(column_data)

In [4]:
allowed = {
    "Food Product Group": [
        "Produce",
        "Condiments & Snacks",
        "Meat",
        "Bread, Grains & Legumes",
        "Meals",
        "Milk & Dairy",
        "Beverages",
        "Non-Food",
        "Seafood",
    ],
    "Food Product Category": [
        "Condiments & Snacks",
        "Vegetables",
        "Meals",
        "Fruit",
        "Grain Products",
        "Beverages",
        "Non-Food",
        "Roots & Tubers",
        "Chicken",
        "Beef",
        "Cheese",
        "Pork",
        "Turkey, Other Poultry",
        "Milk & Dairy",
        "Yogurt",
        "Seafood",
        "Legumes",
        "Milk",
        "Eggs",
        "Tree Nuts & Seeds",
        "Rice",
        "Meat",
        "Butter",
        "Fish (Wild)",
        "Fish (Farm-Raised)",
        "Produce",
    ],
    "Primary Food Product Category": [
        "Condiments & Snacks",
        "Vegetables",
        "Fruit",
        "Grain Products",
        "Beverages",
        "Non-Food",
        "Cheese",
        "Roots & Tubers",
        "Meals",
        "Beef",
        "Chicken",
        "Pork",
        "Turkey, Other Poultry",
        "Milk & Dairy",
        "Seafood",
        "Yogurt",
        "Legumes",
        "Milk",
        "Eggs",
        "Tree Nuts & Seeds",
        "Rice",
        "Butter",
        "Fish (Wild)",
        "Fish (Farm-Raised)",
        "Produce",
        "Meat",
        "Egg",
        "Meats",
    ],
    "Basic Type": df["want_Basic Type"].value_counts().index.tolist(),
    "Flavor/Cut": [
        "flavored",
        "breast",
        "ham",
        "wing",
        "thigh",
        "steak",
        "loin",
        "rib",
        "mix",
        "tenderloin",
        "leg",
        "brisket",
        "chuck",
        "butt",
        "sirloin",
        "shoulder",
        "short rib",
        "bottom round",
        "belly",
        "shank",
        "oxtail",
        "skirt",
        "tri tip",
        "striploin",
        "knuckle",
        "rack",
        "shortloin",
        "cheek",
        "neck",
        "round",
        "tripe",
        "tongue",
        "teres major",
        "pectoral meat",
        "outside skirt",
        "marrow bone",
        "loin rib",
        "t-bone",
        "cut",
    ],
    "Shape": [
        "cut",
        "patty",
        "ground",
        "concentrate",
        "bacon",
        "hot dog",
        "meatball",
        "thickened",
        "crumble",
        "nugget",
        "jerky",
        "salami",
        "pepperoni",
        "pastrami",
        "bologna",
        "prosciutto",
        "shredded",
        "genoa",
        "liquid",
        "mortadella",
        "capocollo",
        "pancetta",
        "bresaola",
        "sopressata",
        "breast",
        "cotto",
        "guanciale",
        "nostrano",
    ],
    "Skin": [
        "skin on",
        "tail on",
        "shell on",
    ],
    "Seed/Bone": [
        "bone-in",
        "pitted",
    ],
    "Processing": [
        "breaded",
        "in juice",
        "seasoned",
        "dried",
        "in syrup",
        "puree",
        "powder",
        "in water",
        "battered",
        "hard boiled",
        "dehydrated",
        "whipped",
        "grated",
        "corned",
        "in sauce",
        "stuffed",
        "in brine",
        "in oil",
        "evaporated",
        "in puree",
        "in vinegar",
        "in liquid",
        "in gel",
        "marinated",
        "powdered",
        "in vegetable broth",
    ],
    "Cooked/Cleaned": [
        "cooked",
        "smoked",
    ],
    "WG/WGR": [
        "whole grain rich",
    ],
    "Dietary Concern": [
        "nonfat",
        "low sodium",
        "low fat",
        "1%",
        "salted",
        "unsalted",
        "decaffeinated",
        "diet",
        "2%",
        "reduced sodium",
        "fat free",
        "reduced sugar",
        "no sodium",
        "reduced calorie",
        "caffeinated",
    ],
    "Additives": [
        "no additives",
        "unsweetened",
        "additives",
        "sweetened",
    ],
    "Dietary Accommodation": [
        "gluten free",
        "kosher",
        "vegan",
        "vegetarian",
        "lactose free",
        "halal",
        "non-dairy",
    ],
    "Frozen": [
        "frozen",
        "iced",
    ],
    "Packaging": [
        "ss",
        "canned",
        "jarred",
    ],
    "Commodity": [
        "commodity",
    ],
}


In [5]:
p_correct = {}

In [6]:
def want_and_got(column):
    mapping = {x: x for x in allowed[column]} | {None: "(missing)", "": "(missing)"}
    want = df[f"want_{column}"].map(lambda x: mapping.get(x, "(malformed)"))
    got = df[f"got_{column}"].map(lambda x: "malformed" if isinstance(x, list) else mapping.get(x, "(malformed)"))
    print("malformed values:")
    print(np.unique(df[f"got_{column}"][got == "(malformed)"]))
    return want, got

In [7]:
def confusion_matrix(column):
    print(column)
    print("-" * 80)

    want, got = want_and_got(column)
    table = pd.crosstab(want, got, margins=True, margins_name="(total)")

    print("-" * 80)
    nottotal = [x for x in table.columns if x != "(total)"]
    probtable = pd.DataFrame({
        "P(output)": [100 * table[x].loc["(total)"] / table["(total)"].loc["(total)"] for x in nottotal],
        "P(correct)": [100 * (0 if x not in table.index else table[x].loc[x]) / table[x].loc["(total)"] for x in nottotal],
    }, index=nottotal)
    print(probtable)
    print("-" * 80)
    overall = 100 * np.count_nonzero(want == got) / len(df)
    print(f"overall P(correct): {overall:9.5f}%")
    print()
    p_correct[column] = {
        "overall": overall,
        "byvalue": {
            "" if index == "(missing)" else index: row["P(correct)"]
            for index, row in probtable.iterrows()
            if index != "(malformed)"
        },
        "numsamples": {
            "" if index == "(missing)" else index: int(row["P(correct)"] * table[index].loc["(total)"] / 100)
            for index, row in probtable.iterrows()
            if index != "(malformed)"
        },
    }

    middle = table[[x for x in table.columns if not x.startswith("(")]].loc[[x for x in table.index if not x.startswith("(")]]

    table.index.name = "want:"
    table.columns.name = "got:"
    return table.style.background_gradient(cmap="Blues", vmin=0, vmax=middle.max().max())

In [8]:
confusion_matrix("Food Product Group")

Food Product Group
--------------------------------------------------------------------------------
malformed values:
['Beans' 'Beef' 'Butter' 'Butter, Eggs & Other' 'Butter, Oil & Condiments'
 'Butter, Other Dairy' 'Cheese' 'Chicken' 'Chicken, Other Poultry'
 'Commodity' 'Dairy' 'Desserts' 'Eggs' 'Fish (Farm-Raised)' 'Fruit'
 'Grain Products' 'Legumes' 'Meet' 'Mine' 'Plant-Based' 'Pork' 'Potatoes'
 'Poultry' 'Protein' 'Protein Foods' 'Rice' 'Roots & Tubers' 'Sandwiches'
 'Soy' 'Tree Nuts & Seeds' 'Turkey, Other Poultry' 'Vegetables' 'WG/WGR']
--------------------------------------------------------------------------------
                         P(output)  P(correct)
(malformed)              15.297998    0.000000
(missing)                 0.004550    0.000000
Beverages                 5.962238   94.353300
Bread, Grains & Legumes   6.813012   94.624374
Condiments & Snacks      26.280710   84.281139
Meals                     6.146497   88.823094
Meat                      3.798908   85.

got:,(malformed),(missing),Beverages,"Bread, Grains & Legumes",Condiments & Snacks,Meals,Meat,Milk & Dairy,Non-Food,Produce,Seafood,(total)
want:,,,,,,,,,,,,
Beverages,9,0,2473,0,63,2,3,30,7,17,0,2604
"Bread, Grains & Legumes",281,0,3,2834,911,84,21,3,4,114,0,4255
Condiments & Snacks,79,1,106,96,9737,153,42,118,50,235,5,10622
Meals,891,0,10,56,357,2400,159,44,7,157,59,4140
Meat,3395,0,0,0,12,22,1435,5,1,3,1,4874
Milk & Dairy,253,1,6,1,116,10,0,2556,3,5,0,2951
Non-Food,4,0,14,8,57,16,6,8,2362,12,3,2490
Produce,1810,0,9,0,300,15,2,3,4,9195,1,11339
Seafood,3,0,0,0,0,0,2,0,0,3,677,685


In [9]:
confusion_matrix("Food Product Category")

Food Product Category
--------------------------------------------------------------------------------
malformed values:
['Beans' 'Bread, Grain & Legumes' 'Bread, Grain Products'
 'Bread, Grain Products & Legumes' 'Bread, Grains & Legumes'
 'Butter, Oil & Condiments' 'Butter, Other Dairy' 'Cereal' 'Commodity'
 'Dessert' 'Desserts' 'Duck' 'Egg' 'Herbs' 'Herbs & Spices' 'Pizza'
 'Plant-Based' 'Potato' 'Potatoes' 'Root & Tubers' 'Seed/Nut' 'Soy'
 'Soy Products' 'Squash' 'Veal' 'WG/WGR']
--------------------------------------------------------------------------------
                       P(output)  P(correct)
(malformed)             0.461783    0.000000
Beef                    4.192448   69.234943
Beverages               5.950864   94.495413
Butter                  0.418562   61.413043
Cheese                  4.488171   62.797770
Chicken                 4.210646   78.660184
Condiments & Snacks    25.618744   86.236903
Eggs                    1.105551   63.991770
Fish (Farm-Raised)      0

got:,(malformed),Beef,Beverages,Butter,Cheese,Chicken,Condiments & Snacks,Eggs,Fish (Farm-Raised),Fish (Wild),Fruit,Grain Products,Legumes,Meals,Meat,Milk,Milk & Dairy,Non-Food,Pork,Produce,Rice,Roots & Tubers,Seafood,Tree Nuts & Seeds,"Turkey, Other Poultry",Vegetables,Yogurt,(total)
want:,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Beef,1,1276,0,0,3,3,1,0,0,0,1,0,0,1,3,0,0,1,5,0,0,0,0,0,3,0,0,1298
Beverages,1,3,2472,0,4,0,61,3,0,0,20,0,0,2,0,12,12,7,0,1,0,1,0,0,0,2,3,2604
Butter,0,0,0,113,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,116
Cheese,0,1,0,0,1239,0,12,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,1255
Chicken,0,1,0,0,1,1456,2,0,0,0,0,0,0,2,0,0,0,0,3,0,0,0,1,0,5,1,0,1472
Condiments & Snacks,18,30,106,59,37,17,9712,4,0,0,47,97,2,130,0,3,64,50,4,24,0,13,5,35,8,155,2,10622
Eggs,0,1,0,0,0,0,0,311,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,314
Fish (Farm-Raised),0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,55,0,0,0,0,60
Fish (Wild),0,0,0,0,0,2,0,0,4,9,0,0,0,0,0,0,0,0,0,0,0,0,106,0,0,0,0,121


In [10]:
confusion_matrix("Primary Food Product Category")

Primary Food Product Category
--------------------------------------------------------------------------------
malformed values:
['Beans' 'Bread, Grain & Legumes' 'Bread, Grain Products'
 'Bread, Grain Products & Legumes' 'Bread, Grains & Legges'
 'Bread, Grains & Legumes' 'Butter, Oil & Condiments'
 'Butter, Other Dairy' 'Commodity' 'Dessert' 'Desserts' 'Duck' 'Herbs'
 'Herbs & Spices' 'Pizza' 'Plant-Based' 'Potato' 'Potatoes' 'Seed/Nut'
 'Soy' 'Soy Products' 'Squash' 'Tuna (Farm-Raised)' 'Veal' 'Veggie'
 'WG/WGR']
--------------------------------------------------------------------------------
                       P(output)  P(correct)
(malformed)             0.420837    0.000000
(missing)               3.560055    0.063898
Beef                    4.292539   90.302067
Beverages               5.773430   94.562648
Butter                  0.386715   62.352941
Cheese                  4.870337   91.219057
Chicken                 4.274340   94.571581
Condiments & Snacks    24.808917   86

got:,(malformed),(missing),Beef,Beverages,Butter,Cheese,Chicken,Condiments & Snacks,Egg,Eggs,Fish (Farm-Raised),Fish (Wild),Fruit,Grain Products,Legumes,Meals,Meat,Milk,Milk & Dairy,Non-Food,Pork,Produce,Rice,Roots & Tubers,Seafood,Tree Nuts & Seeds,"Turkey, Other Poultry",Vegetables,Yogurt,(total)
want:,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
(missing),1,1,4,0,0,0,0,2,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,11
Beef,2,32,1704,0,0,35,9,11,0,0,0,0,1,2,0,14,4,0,1,1,13,0,0,0,0,0,6,3,0,1838
Beverages,1,74,3,2400,0,4,0,60,0,3,0,0,20,0,0,2,0,12,12,6,0,1,0,1,0,0,0,2,3,2604
Butter,0,7,0,0,106,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,116
Cheese,2,19,5,0,0,1953,34,40,3,37,0,0,0,8,0,35,0,0,1,1,4,0,0,1,4,1,81,2,0,2231
Chicken,0,13,3,0,0,1,1777,7,0,0,0,0,0,2,0,7,0,0,0,1,3,0,0,0,1,0,9,2,0,1826
Condiments & Snacks,16,304,31,104,54,51,18,9429,0,4,0,0,47,93,2,108,0,3,60,46,3,23,1,12,5,35,7,155,2,10613
Eggs,0,47,1,0,0,2,0,0,11,315,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,1,0,380
Fish (Farm-Raised),0,1,0,0,0,0,0,0,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,0,53,0,0,0,0,62


In [11]:
confusion_matrix("Basic Type")
None

Basic Type
--------------------------------------------------------------------------------
malformed values:
['addon' 'alcohol' 'alcoholic beverage' 'almond' 'anise' 'apple butter'
 'apple sauce' 'artichoke heart' 'auce' 'bas' 'bean salad' 'beans'
 'beef alternative' 'beef chili' 'beef frank' 'beef sub' 'beef substitute'
 'beefaroni' 'beefless' 'beer' 'bell pepper' 'beverage' 'bicarbonate'
 'black bean' 'blend' 'bouillon' 'bread pudding' 'broccoli slaw' 'burgers'
 'cake' 'calzonette' 'candy bar' 'cannoli' 'cantaloupe' 'caperberry'
 'capers' 'cheeseburger' 'cheesecake' 'chicken alternative' 'chili pepper'
 'chili relleno' 'chimi' 'chipotle' 'chipotle sauce' 'chive' 'chocolate'
 'cinnamon raisin' 'cinnamon toast' 'coating' 'cocktail' 'cocoa'
 'cocoa mix' 'coffee bean' 'cola' 'coleslaw' 'collard' 'coloring'
 'corn chip' 'corn meal' 'corn nut' 'cornflake' 'cornish hen' 'cornstarch'
 'cotton candy' 'covering' 'crackers' 'craisins' 'cranberry dried'
 'crayfish' 'crema' 'crumb' 'crumble' 'cu

In [12]:
confusion_matrix("Flavor/Cut")

Flavor/Cut
--------------------------------------------------------------------------------
malformed values:
['all' 'bacon' 'banana' 'barbecue' 'base' 'beef' 'blueberry' 'bone-in'
 'brat' 'buttermilk' 'cap' 'capicola' 'cheese' 'chicken' 'chip'
 'chocolate' 'chop' 'clod' 'cream' 'cup' 'dark' 'dark meat' 'drum'
 'drumette' 'drumstick' 'fajita' 'feet' 'flank' 'flank steak' 'flap'
 'flat iron' 'foot' 'grated' 'hanger' 'heart' 'hindshank' 'honey'
 'hot dog' 'iced' 'inch' 'inside round' 'inside skirt' 'link' 'marinated'
 'no sodium' 'ox tail' 'pepperoni' 'picnic' 'pork' 'prime rib' 'pulled'
 'quarter' 'ribeye' 'roast' 'salted' 'sausage' 'seasoned' 'sharp'
 'shredded' 'skin on' 'smoked' 'spicy' 'strawberry' 'stuffed' 'substitute'
 'tail' 'tender' 'thickened' 'tip' 'top inside round' 'top round'
 'top sirloin' 'vanilla' 'variety']
--------------------------------------------------------------------------------
               P(output)  P(correct)
(malformed)     0.520928    0.000000
(missing)

got:,(malformed),(missing),belly,bottom round,breast,brisket,butt,cheek,chuck,cut,flavored,ham,knuckle,leg,loin,malformed,mix,neck,outside skirt,oxtail,pectoral meat,rack,rib,round,shank,short rib,shortloin,shoulder,sirloin,skirt,steak,striploin,tenderloin,teres major,thigh,tongue,tri tip,wing,(total)
want:,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
(missing),181,38537,0,15,72,6,2,0,5,42,251,28,0,10,19,2,20,1,0,0,0,0,15,9,0,0,0,0,3,2,17,1,20,2,8,0,1,2,39271
belly,0,7,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13
bottom round,0,9,0,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18
breast,1,54,0,0,554,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,5,0,0,0,0,0,617
brisket,0,4,0,0,0,45,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,49
butt,1,10,0,0,0,0,36,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,48
cheek,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5
chuck,1,16,0,0,0,0,0,1,34,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,54
flavored,11,2365,0,0,1,1,0,0,0,6,477,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2863


In [13]:
confusion_matrix("Shape")

Shape
--------------------------------------------------------------------------------
malformed values:
['aroma' 'bagel' 'bar' 'battered' 'beef' 'belly' 'breaded' 'bun' 'cake'
 'cheese' 'chewy' 'chip' 'coarse' 'cone' 'corn dog' 'corned' 'croissant'
 'crumb' 'crumbled' 'crushed' 'crust' 'cup' 'curd' 'cut. Cooked' 'dough'
 'doughnut' 'dried' 'drumstick' 'eggs' 'empanada' 'frank' 'fritter'
 'grated' 'ham' 'hamburger' 'hash' 'hash brown' 'iced' 'in water' 'juice'
 'knockwurst' 'link' 'mashed' 'meat' 'meatloaf' 'mix' 'noodle' 'pancake'
 'pasta' 'paste' 'pie' 'pork' 'portion' 'powder' 'protein' 'puree' 'rib'
 'rip' 'roll' 'salad' 'sandwich' 'sausage' 'scramble' 'scrambled'
 'seasoned' 'short rib' 'skin on' 'sparerib' 'ss' 'steak' 'stick'
 'stuffed' 'sub' 'summer sausage' 'taco' 'taquito' 'tater tot'
 'tenderloin' 'wafer' 'whipped' 'wing']
--------------------------------------------------------------------------------
             P(output)  P(correct)
(malformed)   0.723385    0.000000
(mi

got:,(malformed),(missing),bacon,bologna,bresaola,capocollo,concentrate,crumble,cut,ground,hot dog,jerky,liquid,meatball,mortadella,nugget,pastrami,patty,pepperoni,prosciutto,salami,shredded,thickened,(total)
want:,,,,,,,,,,,,,,,,,,,,,,,,
(missing),285,34187,12,1,0,1,39,11,1221,172,84,22,5,12,0,21,0,176,4,0,4,8,5,36270
bacon,0,42,140,0,0,0,0,1,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,189
bologna,0,1,0,17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18
breast,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
bresaola,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
capocollo,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
concentrate,1,260,0,0,0,0,15,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,278
cotto,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
crumble,2,21,0,0,0,0,0,57,3,7,0,0,0,0,0,0,0,0,0,0,0,0,0,90


In [14]:
confusion_matrix("Skin")

Skin
--------------------------------------------------------------------------------
malformed values:
['casing' 'casings in' 'flavored' 'seeded' 'skinless']
--------------------------------------------------------------------------------
             P(output)  P(correct)
(malformed)   0.043221    0.000000
(missing)    99.410828   99.915334
shell on      0.009099    0.000000
skin on       0.500455   32.727273
tail on       0.036397   56.250000
--------------------------------------------------------------------------------
overall P(correct):  99.51092%



got:,(malformed),(missing),shell on,skin on,tail on,(total)
want:,,,,,,
(missing),19,43664,4,148,7,43842
shell on,0,1,0,0,0,1
skin on,0,27,0,72,0,99
tail on,0,9,0,0,9,18
(total),19,43701,4,220,16,43960


In [15]:
confusion_matrix("Seed/Bone")

Seed/Bone
--------------------------------------------------------------------------------
malformed values:
['boneless' 'natural casing' 'seeded' 'skinless' 'tail on']
--------------------------------------------------------------------------------
             P(output)  P(correct)
(malformed)   0.036397    0.000000
(missing)    99.372157   99.935903
bone-in       0.323021   29.577465
pitted        0.268426   25.423729
--------------------------------------------------------------------------------
overall P(correct):  99.47225%



got:,(malformed),(missing),bone-in,pitted,(total)
want:,,,,,
(missing),16,43656,100,88,43860
bone-in,0,22,42,0,64
pitted,0,6,0,30,36
(total),16,43684,142,118,43960


In [16]:
confusion_matrix("Processing")

Processing
--------------------------------------------------------------------------------
malformed values:
['additives' 'aged' 'baked' 'barbecue' 'blanched' 'blend' 'canned'
 'chilled' 'classic' 'concentrate' 'condensed' 'cook' 'cooked' 'creamed'
 'crumb' 'crumbled' 'crushed' 'cured' 'cut' 'decaffeinated' 'diluted'
 'dry' 'enriched' 'extra firm' 'floret' 'fresh' 'fried' 'frozen'
 'gluten free' 'grilled' 'ground' 'homogenized' 'iced' 'in broth'
 'in chocolate' 'in concentrate' 'in cream' 'in dressing' 'in gravy'
 'in jelly' 'in liquor' 'in mayonnaise' 'in shell' 'in solution' 'instant'
 'jerky' 'liquid' 'low fat' 'low moisture' 'low sodium' 'mashed'
 'mechanically separated' 'mix' 'omelette' 'par boiled' 'parbaked'
 'parboiled' 'partially cooked' 'pasteurized' 'pastrami' 'peeled'
 'pre-fried' 'precooked' 'precut' 'pressed' 'processed' 'prosciutto'
 'ready to eat' 'reduced calorie' 'reduced sugar' 'refried' 'rendered'
 'roasted' 'salted' 'sauce' 'scrambled' 'seedless' 'serving' 'shred

got:,(malformed),(missing),battered,breaded,corned,dehydrated,dried,evaporated,grated,hard boiled,in brine,in juice,in liquid,in oil,in puree,in sauce,in syrup,in vegetable broth,in vinegar,in water,marinated,powder,powdered,puree,seasoned,stuffed,whipped,(total)
want:,,,,,,,,,,,,,,,,,,,,,,,,,,,,
(missing),438,39679,22,105,6,13,264,0,5,3,29,146,3,14,12,86,36,1,6,66,3,100,14,73,488,14,4,41630
battered,0,18,44,22,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,87
breaded,0,154,7,475,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,641
corned,0,5,0,0,32,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,37
dehydrated,0,7,0,0,0,19,22,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,0,0,51
dried,0,79,0,0,0,0,174,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,255
evaporated,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,9
grated,0,13,0,0,0,0,0,0,37,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,50
hard boiled,0,51,0,0,0,0,1,0,0,12,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65


In [17]:
confusion_matrix("Cooked/Cleaned")

Cooked/Cleaned
--------------------------------------------------------------------------------
malformed values:
['breaded' 'cleaned' 'dehydrated' 'fried' 'hard boiled' 'iced' 'parfried'
 'pasteurized' 'raw' 'ready to eat' 'seasoned']
--------------------------------------------------------------------------------
             P(output)  P(correct)
(malformed)   0.040946    0.000000
(missing)    94.071884   98.551531
cooked        5.168335   59.771127
smoked        0.718835   51.265823
--------------------------------------------------------------------------------
overall P(correct):  96.16697%



got:,(malformed),(missing),cooked,smoked,(total)
want:,,,,,
(missing),15,40755,911,111,41792
cooked,3,566,1358,43,1970
smoked,0,33,3,162,198
(total),18,41354,2272,316,43960


In [18]:
confusion_matrix("WG/WGR")

WG/WGR
--------------------------------------------------------------------------------
malformed values:
['1%' 'ss']
--------------------------------------------------------------------------------
                  P(output)  P(correct)
(malformed)        0.009099    0.000000
(missing)         94.717925   98.508574
whole grain rich   5.272975   82.786885
--------------------------------------------------------------------------------
overall P(correct):  97.67061%



got:,(malformed),(missing),whole grain rich,(total)
want:,,,,
(missing),4,41017,399,41420
whole grain rich,0,621,1919,2540
(total),4,41638,2318,43960


In [19]:
confusion_matrix("Dietary Concern")

Dietary Concern
--------------------------------------------------------------------------------
malformed values:
['10%' '3% milkfat' 'additives' 'dietary accommodation' 'dieted'
 'dietetic' 'extra lean' 'full fat' 'gluten free' 'kosher' 'lactose free'
 'lean' 'low calorie' 'low milkfat' 'low moisture' 'low sugar' 'no sugar'
 'reduced' 'reduced fat' 'salt free' 'special diet' 'unsweetened' 'whole'
 'whole milk' 'zero trans fat']
--------------------------------------------------------------------------------
                 P(output)  P(correct)
(malformed)       0.152411    0.000000
(missing)        95.907643   99.255236
1%                0.373066   74.390244
2%                0.225205   83.838384
caffeinated       0.006824    0.000000
decaffeinated     0.172884   78.947368
diet              0.186533   40.243902
fat free          0.100091   20.454545
low fat           0.605096   47.368421
low sodium        0.846224   62.903226
no sodium         0.061419    3.703704
nonfat           

got:,(malformed),(missing),1%,2%,caffeinated,decaffeinated,diet,fat free,low fat,low sodium,no sodium,nonfat,reduced calorie,reduced sodium,reduced sugar,salted,unsalted,(total)
want:,,,,,,,,,,,,,,,,,,
(missing),66,41847,18,12,3,16,46,17,128,122,23,54,9,22,11,22,33,42449
1%,0,2,122,1,0,0,0,0,1,0,0,0,0,0,0,0,0,126
2%,0,1,4,83,0,0,0,0,0,0,0,0,0,0,0,0,0,88
caffeinated,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4
decaffeinated,0,15,0,0,0,60,1,0,0,0,0,1,0,0,0,0,0,77
diet,0,44,0,0,0,0,33,0,0,0,0,0,0,0,0,0,0,77
fat free,0,4,0,0,0,0,1,9,0,0,0,16,0,0,0,0,0,30
low fat,1,17,11,3,0,0,0,0,126,0,0,2,0,0,0,0,0,160
low sodium,0,51,0,0,0,0,0,0,1,234,0,0,0,8,0,0,0,294


In [20]:
confusion_matrix("Additives")

Additives
--------------------------------------------------------------------------------
malformed values:
['reduced' 'salted' 'unsalted']
--------------------------------------------------------------------------------
              P(output)  P(correct)
(malformed)    0.015924    0.000000
(missing)     98.682894   99.183975
additives      0.111465    0.000000
no additives   0.348044    0.653595
sweetened      0.052320    4.347826
unsweetened    0.789354   26.224784
--------------------------------------------------------------------------------
overall P(correct):  98.08917%



got:,(malformed),(missing),additives,no additives,sweetened,unsweetened,(total)
want:,,,,,,,
(missing),7,43027,47,148,22,250,43501
additives,0,48,0,0,0,1,49
no additives,0,220,1,1,0,5,227
sweetened,0,2,1,0,1,0,4
unsweetened,0,84,0,4,0,91,179
(total),7,43381,49,153,23,347,43960


In [21]:
confusion_matrix("Dietary Accommodation")

Dietary Accommodation
--------------------------------------------------------------------------------
malformed values:
['decaffeinated' 'diet' 'soy free']
--------------------------------------------------------------------------------
              P(output)  P(correct)
(malformed)    0.006824    0.000000
(missing)     98.491811   99.538074
gluten free    0.743858   64.831804
halal          0.104641   82.608696
kosher         0.266151   52.136752
lactose free   0.054595   79.166667
non-dairy      0.029572    0.000000
vegan          0.161510   42.253521
vegetarian     0.141037   43.548387
--------------------------------------------------------------------------------
overall P(correct):  98.91720%



got:,(malformed),(missing),gluten free,halal,kosher,lactose free,non-dairy,vegan,vegetarian,(total)
want:,,,,,,,,,,
(missing),3,43097,114,8,55,3,13,37,35,43365
gluten free,0,76,212,0,1,2,0,2,0,293
halal,0,3,0,38,0,0,0,0,0,41
kosher,0,19,0,0,61,0,0,0,0,80
lactose free,0,20,0,0,0,19,0,0,0,39
non-dairy,0,3,0,0,0,0,0,0,0,3
vegan,0,39,1,0,0,0,0,30,0,70
vegetarian,0,40,0,0,0,0,0,2,27,69
(total),3,43297,327,46,117,24,13,71,62,43960


In [22]:
confusion_matrix("Frozen")

Frozen
--------------------------------------------------------------------------------
malformed values:
[]
--------------------------------------------------------------------------------
           P(output)  P(correct)
(missing)  92.493176   99.001476
frozen      7.158781   68.541468
iced        0.348044   18.300654
--------------------------------------------------------------------------------
overall P(correct):  96.54004%



got:,(missing),frozen,iced,(total)
want:,,,,
(missing),40254,988,122,41364
frozen,370,2157,3,2530
iced,36,2,28,66
(total),40660,3147,153,43960


In [23]:
confusion_matrix("Packaging")

Packaging
--------------------------------------------------------------------------------
malformed values:
[]
--------------------------------------------------------------------------------
           P(output)  P(correct)
(missing)  86.803913   93.870384
canned      1.710646   69.946809
jarred      0.020473    0.000000
ss         11.464968   68.313492
--------------------------------------------------------------------------------
overall P(correct):  90.51183%



got:,(missing),canned,jarred,ss,(total)
want:,,,,,
(missing),35820,226,9,1478,37533
canned,283,526,0,115,924
jarred,9,0,0,4,13
ss,2047,0,0,3443,5490
(total),38159,752,9,5040,43960


In [24]:
confusion_matrix("Commodity")

Commodity
--------------------------------------------------------------------------------
malformed values:
[]
--------------------------------------------------------------------------------
           P(output)  P(correct)
(missing)  99.160601   99.479250
commodity   0.839399   42.276423
--------------------------------------------------------------------------------
overall P(correct):  98.99909%



got:,(missing),commodity,(total)
want:,,,
(missing),43364,213,43577
commodity,227,156,383
(total),43591,369,43960


In [25]:
print(json.dumps(p_correct, indent=4))

{
    "Food Product Group": {
        "overall": 76.59008189262967,
        "byvalue": {
            "": 0.0,
            "Beverages": 94.35330026707364,
            "Bread, Grains & Legumes": 94.62437395659433,
            "Condiments & Snacks": 84.28113909806977,
            "Meals": 88.82309400444116,
            "Meat": 85.92814371257485,
            "Milk & Dairy": 92.37441272135887,
            "Non-Food": 96.88269073010665,
            "Produce": 94.39482599322452,
            "Seafood": 90.75067024128687
        },
        "numsamples": {
            "": 0,
            "Beverages": 2473,
            "Bread, Grains & Legumes": 2834,
            "Condiments & Snacks": 9737,
            "Meals": 2400,
            "Meat": 1435,
            "Milk & Dairy": 2555,
            "Non-Food": 2362,
            "Produce": 9195,
            "Seafood": 677
        }
    },
    "Food Product Category": {
        "overall": 83.13694267515923,
        "byvalue": {
            "Beef": 69.23494302

In [26]:
basic_types = ["(malformed)"] + allowed["Basic Type"]
toindex = {x: i for i, x in enumerate(basic_types)}

good_good = np.zeros(len(toindex) * 2, dtype=int)
good_wantmore = np.zeros(len(toindex) * 2, dtype=int)
gotmore_good = np.zeros(len(toindex) * 2, dtype=int)
gotmore_wantmore = np.zeros(len(toindex) * 2, dtype=int)
denominator = np.zeros(len(toindex) * 2, dtype=int)

for _, row in df.iterrows():
    want = set(row["want_Sub-Type"] or [])
    got = set(row["got_Sub-Type"] or [])
    index = toindex.get(row["got_Basic Type"], 0) * 2 + int(len(got) == 0)

    wantmore = not want.issubset(got)
    gotmore = not got.issubset(want)
    if not wantmore and not gotmore:
        good_good[index] += 1
    if wantmore and not gotmore:
        good_wantmore[index] += 1
    if not wantmore and gotmore:
        gotmore_good[index] += 1
    if wantmore and gotmore:
        gotmore_wantmore[index] += 1
    denominator[index] += 1

table = pd.DataFrame({
    "good_good": good_good,
    "good_wantmore": good_wantmore,
    "gotmore_good": gotmore_good,
    "gotmore_wantmore": gotmore_wantmore,
    "denominator": denominator,
}, index=pd.MultiIndex.from_product([pd.Index(basic_types, name="Basic Type"), pd.Index([False, True], name="empty Sub-Type")]))

In [27]:
table

good_good  good_wantmore  gotmore_good  \
Basic Type  empty Sub-Type                                           
(malformed) False                  45             19            98   
            True                 4500           1231             0   
chicken     False                  46              9            71   
            True                 1119            303             0   
beef        False                  50             17            81   
...                               ...            ...           ...   
scallion    True                    3              0             0   
bakery      False                   0              0             0   
            True                    0              0             0   
sea bass    False                   0              0             0   
            True                    0              0             0   

                            gotmore_wantmore  denominator  
Basic Type  empty Sub-Type                                 
(malformed) False                        117          279  
            True                           0         5731  
chicken     False                         72          198  
            True                           0         1422  
beef        False                         67          215  
...                                      ...          ...  
scallion    True                           0            3  
bakery      False                          0            0  
            True                           0            0  
sea bass    False                          0            0  
            True                           0            0  

[1120 rows x 5 columns]

In [28]:
(table["good_good"] + table["good_wantmore"] + table["gotmore_good"] + table["gotmore_wantmore"] == table["denominator"]).all()

np.True_

In [29]:
prepare = (table["good_good"] / table["denominator"] * 100).to_frame("P(correct)").assign(numsamples=table["denominator"])[table["denominator"] != 0]

In [30]:
prepare

P(correct)  numsamples
Basic Type  empty Sub-Type                        
(malformed) False            16.129032         279
            True             78.520328        5731
chicken     False            23.232323         198
            True             78.691983        1422
beef        False            23.255814         215
...                                ...         ...
seafood     False            33.333333           6
            True             66.666667           3
frosting    True            100.000000           1
ti leaf     True            100.000000           1
scallion    True            100.000000           3

[704 rows x 2 columns]

In [31]:
p_subtype_correct = {
    "overall": table["good_good"].sum() / table["denominator"].sum() * 100,
    "nonempty_Sub-Type": table["good_good"].loc[:, False].sum() / table["denominator"].loc[:, False].sum() * 100,
    "empty_Sub-Type": table["good_good"].loc[:, True].sum() / table["denominator"].loc[:, True].sum() * 100,
    "byvalue_nonempty": dict(prepare["P(correct)"].loc[:, False]),
    "byvalue_empty": dict(prepare["P(correct)"].loc[:, True]),
    "numsamples_nonempty": {k: int(v) for k, v in dict(prepare["numsamples"].loc[:, False]).items()},
    "numsamples_empty": {k: int(v) for k, v in dict(prepare["numsamples"].loc[:, True]).items()},
}

In [32]:
print(json.dumps(p_subtype_correct, indent=4))

{
    "overall": 62.93221110100091,
    "nonempty_Sub-Type": 50.863821138211385,
    "empty_Sub-Type": 72.71416803953872,
    "byvalue_nonempty": {
        "(malformed)": 16.129032258064516,
        "chicken": 23.232323232323232,
        "beef": 23.25581395348837,
        "cheese": 56.93860386879731,
        "condiment": 58.17774458551157,
        "juice": 30.702598652550527,
        "pork": 60.42553191489362,
        "sauce": 59.62199312714777,
        "dessert": 52.43445692883895,
        "pepper": 73.57142857142858,
        "potato": 52.81249999999999,
        "turkey": 42.857142857142854,
        "seasoned": 72.50608272506082,
        "apple": 76.28205128205127,
        "cereal": 39.65936739659367,
        "chip": 42.26086956521739,
        "tomato": 75.77319587628865,
        "carrot": 16.666666666666664,
        "yogurt": 20.462046204620464,
        "dressing": 75.1578947368421,
        "lettuce": 73.9938080495356,
        "onion": 57.14285714285714,
        "cracker": 40.2402402